# 11. クロネッカー積・テンソル・行列微分(付録)— ML への橋

| 層 | セクション |
|---|---|
| Basic | 1. Big Picture 〜 3. vec とクロネッカー |
| Applied | 4. テンソル 〜 5. 行列微分 |
| Advanced | 6. 数値検証 / 7. Advanced Notes |

## 1. Big Picture

機械学習のコードを読むと、**クロネッカー積**・**vec 演算**・**行列微分** が至る所に出てきます。
本章はこの 3 つを、これまでの線形代数(行列 = 線形変換、04 章の最小二乗、06 章の勾配)の
延長として導入し、ニューラルネット教材(逆伝播 = 行列微分)への橋を架けます。

```{admonition} 核心 — ひとことで
:class: tip
**クロネッカー積・vec・行列微分は、行列の問題を「巨大だが普通のベクトル方程式・勾配」に翻訳する道具である。**
$\mathrm{vec}(AXB)=(B^\top\otimes A)\,\mathrm{vec}(X)$ で行列方程式は $Ax=b$（02 章）に化け、
$\partial(x^\top Ax)/\partial x = 2Ax$ などの行列微分は、最小二乗（04 章）・勾配降下（06 章）・逆伝播の共通言語になる。
```

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 2. クロネッカー積 — 行列の「ブロック展開」

クロネッカー積 $A \otimes B$ は、$A$ の各成分 $a_{ij}$ を $a_{ij} B$ というブロックに置き換えたもの。

$$
A \otimes B = \begin{pmatrix} a_{11} B & \cdots & a_{1n} B \\ \vdots & & \vdots \\ a_{m1} B & \cdots & a_{mn} B \end{pmatrix}
$$

$A: m \times n$, $B: p \times q$ なら $A \otimes B: mp \times nq$。画像の 2 次元畳み込み・
共分散のクロネッカー構造・量子の合成系などで現れます。

In [2]:
# Kronecker product replaces each entry a_ij of A with the block a_ij * B.
A = np.array([[1.0, 2.0], [0.0, 3.0]])
B = np.array([[1.0, 1.0], [0.0, 1.0]])
K = np.kron(A, B)
print("A (2x2) kron B (2x2) -> shape", K.shape)
print(K)

A (2x2) kron B (2x2) -> shape (4, 4)
[[1. 1. 2. 2.]
 [0. 1. 0. 2.]
 [0. 0. 3. 3.]
 [0. 0. 0. 3.]]


### ▶ 触って確かめる — クロネッカー積のブロック構造

$A\otimes B$ は「$A$ の各成分 $a_{ij}$ を、ブロック $a_{ij}B$ に置き換えた」行列です。
スライダーで $A$ の成分を 1 つずつ選ぶと、対応するブロックが黒枠でハイライトされます。
$a_{ij}$ が大きいほどそのブロックの色が濃くなる（＝$B$ を $a_{ij}$ 倍した中身）ことを確かめてください。

In [3]:
# Highlight each block a_ij * B of the Kronecker product A (x) B.
import plotly.io as pio

from la_book import plotting

pio.renderers.default = "plotly_mimetype+notebook_connected"
fig = plotting.plotly_kron_blocks(A, B)
fig.show()

## 3. vec 演算とクロネッカーの魔法

**vec** は行列の列を縦に積み上げてベクトルにする演算。クロネッカー積と組むと、
行列方程式を「普通の」線形方程式に直せます(ML の核心的な恒等式):

$$
\mathrm{vec}(A X B) = (B^\top \otimes A)\, \mathrm{vec}(X)
$$

「行列に挟まれた未知行列 $X$ について解く」問題が、$\mathrm{vec}(X)$ についての $Ax=b$(02 章)になります。

In [4]:
# The vec trick: vec(A X B) = (B^T kron A) vec(X). Verify numerically.
def vec(M):
    return M.flatten(order="F")        # column-major stacking

A = rng.standard_normal((3, 2))
X = rng.standard_normal((2, 4))
B = rng.standard_normal((4, 3))
lhs = vec(A @ X @ B)
rhs = np.kron(B.T, A) @ vec(X)
print("max |vec(AXB) - (B^T kron A) vec(X)| =", np.abs(lhs - rhs).max())

max |vec(AXB) - (B^T kron A) vec(X)| = 4.440892098500626e-16


## 4. テンソルと縮約 — 多次元への一般化

ベクトル(1 階)・行列(2 階)の先が **テンソル**(多階の数の配列)。
行列積は「添字の縮約(和)」の一例で、`np.einsum` で一般化できます。
深層学習のバッチ計算(バッチ × 系列 × 特徴)はテンソル演算そのものです。

In [5]:
# einsum expresses matrix multiply as index contraction, and generalizes it.
A = rng.standard_normal((4, 3))
B = rng.standard_normal((3, 5))
print("matmul == einsum('ij,jk->ik'):", np.allclose(A @ B, np.einsum("ij,jk->ik", A, B)))

# Batched matmul: (batch, m, k) x (batch, k, n) -> (batch, m, n) (as in NN layers).
batch_A = rng.standard_normal((10, 4, 3))
batch_B = rng.standard_normal((10, 3, 5))
batched = np.einsum("bij,bjk->bik", batch_A, batch_B)
print("batched matmul shape:", batched.shape)

matmul == einsum('ij,jk->ik'): True
batched matmul shape: (10, 4, 5)


## 5. 行列微分 — 逆伝播の正体

スカラー関数を行列・ベクトルで微分する規則は、ニューラルネットの勾配計算そのものです。
よく使う 3 つ($A$ は対称とする):

$$
\frac{\partial (a^\top x)}{\partial x} = a, \qquad
\frac{\partial (x^\top A x)}{\partial x} = 2 A x, \qquad
\frac{\partial \|Ax - b\|^2}{\partial x} = 2 A^\top (Ax - b)
$$

3 番目は 04 章の最小二乗の正規方程式($\nabla = 0$ で $A^\top A x = A^\top b$)そのもの。
6 章の勾配降下も、neural_net 教材の逆伝播も、すべてこの行列微分の連鎖です。

In [6]:
# Analytic matrix-calculus gradients (the formulas above).
A = rng.standard_normal((5, 3))
b = rng.standard_normal(5)
x = rng.standard_normal(3)
S = A.T @ A                      # symmetric

grad_quadratic = 2 * S @ x                      # d/dx x^T S x
grad_lsq = 2 * A.T @ (A @ x - b)                # d/dx ||Ax - b||^2
print("grad x^T S x   :", grad_quadratic)
print("grad ||Ax-b||^2:", grad_lsq)

grad x^T S x   : [-2.3397  2.1007 -9.1213]
grad ||Ax-b||^2: [-1.5453 -1.6752 -8.7894]


### ▶ 触って確かめる — 行列微分が動かす勾配降下

行列微分 $\partial\!\left(\tfrac12 x^\top A x - b^\top x\right)/\partial x = Ax-b$ が、そのまま勾配降下の更新方向です。
異方的な二次の谷の等高線上で、スライダーで 1 ステップずつ降下を辿れます。
固有値の比（条件数）が大きいほどジグザグして遅くなる — 06 章の挙動が、行列微分の式から再現されます。

In [7]:
# The matrix-calculus gradient Ax - b is exactly the descent direction.
fig = plotting.plotly_gradient_descent_quadratic()
fig.show()

## 6. 数値微分で検算する

行列微分の公式は、中心差分(02 章・neural_net 教材と同じ手法)で必ず確かめられます。

In [8]:
# Gradient check: analytic vs central-difference (the habit from every chapter).
def numeric_grad(f, x, h=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = h
        g[i] = (f(x + e) - f(x - e)) / (2 * h)
    return g

f_quad = lambda x: x @ S @ x
f_lsq = lambda x: np.sum((A @ x - b) ** 2)
print("quadratic: max |analytic - numeric| =", np.abs(grad_quadratic - numeric_grad(f_quad, x)).max())
print("lsq      : max |analytic - numeric| =", np.abs(grad_lsq - numeric_grad(f_lsq, x)).max())

quadratic: max |analytic - numeric| = 5.357279064810427e-10
lsq      : max |analytic - numeric| = 2.401764120918415e-09


```{admonition} 実社会では
:class: important
この章の 3 点セットは、機械学習フレームワークの内部そのものです。

- 自動微分（PyTorch / JAX）：逆伝播は行列微分の連鎖律を機械化したもの。勾配チェック（数値微分）は実装の必須検査。
- テンソル演算：`einsum` とバッチ matmul が、層・アテンションの計算の中身。
- クロネッカー構造：2 次元畳み込み、共分散の Kronecker 近似（K-FAC 最適化）、量子の合成系で多用。

「行列を巨大なベクトル方程式に直す」発想が、線形代数と深層学習をつなぐ最後の橋です。
```

## 7. まとめ

- **クロネッカー積** $A \otimes B$ は各成分をブロックに展開。$\mathrm{vec}(AXB) = (B^\top \otimes A)\mathrm{vec}(X)$ で
  行列方程式を $Ax=b$(02 章)に帰着できる。
- **テンソル**はベクトル・行列の多階一般化。`einsum` の添字縮約が行列積を含む統一言語で、
  深層学習のバッチ計算の土台。
- **行列微分**($\partial x^\top A x/\partial x = 2Ax$ など)は最小二乗(04 章)・勾配降下(06 章)・
  逆伝播(neural_net 教材)の共通言語。数値微分でいつでも検算できる。

## 8. Exercises

1. $(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$(mixed product property)を数値で確認せよ。
2. $\mathrm{vec}(AX) = (I \otimes A)\mathrm{vec}(X)$ を確かめよ($B = I$ の特別な場合)。
3. `einsum` でトレース $\mathrm{tr}(AB)$ を 1 行で書け(ヒント: `'ij,ji->'`)。
4. $\partial \mathrm{tr}(A^\top X)/\partial X = A$ を数値微分で確認せよ。
5. (発展)2 層線形ネット $y = W_2 W_1 x$ の損失 $\|y - t\|^2$ の $W_1$ に関する勾配を
   行列微分で導き、数値微分と照合せよ(neural_net 教材 03 章の逆伝播)。